In [1]:
# notebook to compile all of the csvs into a single array
import os
import glob
import pandas as pd
import numpy as np
import cantera as ct

In [2]:
# mech_dir = '/scratch/harris.se/guassian_scratch/dib/RMG_MAX/DIB_20241219'

# # mech_dir = '/scratch/harris.se/guassian_scratch/dib/RMG_min/DIB_20241219'

# mech_dir = '/scratch/harris.se/guassian_scratch/dib/RMG_MAX/RMG_MAX_1_20250117'
# # mech_dir = '/scratch/harris.se/guassian_scratch/dib/RMG_min/RMG_min_1_202501017'

# mech_dir = '/work/westgroup/harris.se/autoscience/fuels/dib/RMG_MAX/RMG_MAX_1_20250117'


# mech_dir = '/scratch/harris.se/guassian_scratch/dib/RMG_MAX/RMG_MAX_2_20250119'
mech_dir = '/scratch/harris.se/guassian_scratch/dib/RMG_min/RMG_min_2_20250122'

gas = ct.Solution(os.path.join(mech_dir, 'chem_annotated.yaml'))

In [3]:
# debug this. Why does Neon have any sensitivity at all?

N = 51
test_sp_file = os.path.join(mech_dir, 'table_0024', 'spec_delay_0024_0001.npy')
test_data = np.load(test_sp_file)

# base_delay_file = '/scratch/harris.se/guassian_scratch/dib/RMG_min/DIB_20241219/table_0024/base_delays_0024.npy'
# base_delays = np.load(base_delay_file)

# total_base_delays = np.load(os.path.join(mech_dir, 'total_base_delays.npy'))
# all_delays = np.load(os.path.join(mech_dir, 'total_perturbed_mech_delays.npy'))

In [4]:
test_data.shape

(51, 5)

In [12]:
# compile everything into a big matrix

#             table24 Pressure and phi/ variable temperatures
# species 1
# species 2
# .........
# species N
# reaction 1
# reaction 2
# .........
# reaction M


N = 51
test_sp_file = os.path.join(mech_dir, 'table_0024', 'spec_delay_0024_0000.npy')
test_data = np.load(test_sp_file)
assert N == len(test_data)


N_REACTIONS_PER_FILE = 10
test_rxn_file = os.path.join(mech_dir, 'table_0024', 'reaction_delays_0024_0000.npy')
test_data = np.load(test_rxn_file)

assert N_REACTIONS_PER_FILE * 5 == np.sum(np.sum(np.abs(test_data), axis=1) != 0)  # only 10 nonzero rows...

all_delays = np.zeros((gas.n_species + gas.n_reactions, N, 5))


for i in range(gas.n_species):
    sp_file = os.path.join(mech_dir, 'table_0024', f'spec_delay_0024_{i:04}.npy')
    if not os.path.exists(sp_file):
        print(f'Missing species {i}')
        continue
    all_delays[i, :, :] = np.load(sp_file)
    if np.all(all_delays[i, :, :] == 0):
        print(f'Species {i} is all zero')

# The reaction delay file is just n_reactions x n_temperatures
for i in range(int(gas.n_reactions / N_REACTIONS_PER_FILE) + 1):
    i_start = i * N_REACTIONS_PER_FILE
    i_end = min(((i + 1) * N_REACTIONS_PER_FILE), gas.n_reactions)

    rxn_file = os.path.join(mech_dir, 'table_0024', f'reaction_delays_0024_{i_start:04}.npy')
    if not os.path.exists(rxn_file):
        print(f'Missing reaction file {i_start:04}')
        continue

    all_delays[i_start + gas.n_species : i_end + gas.n_species, :, :] = np.load(rxn_file)[i_start : i_end, :, :]
    if np.all(all_delays[i_start + gas.n_species : i_end + gas.n_species, :, :] == 0):
        print(f'Reaction {i_start} is all zero')
    

Reaction 140 is all zero
Reaction 200 is all zero
Reaction 620 is all zero
Reaction 800 is all zero
Reaction 810 is all zero
Reaction 820 is all zero
Reaction 830 is all zero
Reaction 960 is all zero
Reaction 970 is all zero
Reaction 1000 is all zero
Reaction 1070 is all zero
Reaction 1080 is all zero
Reaction 1210 is all zero
Reaction 1260 is all zero
Reaction 1280 is all zero


In [6]:
np.load(os.path.join(mech_dir, 'table_0024', f'reaction_delays_0024_{0:04}.npy')).shape

(1283, 51, 5)

In [7]:
gas.n_species

301

In [13]:
all_delays.shape

(1584, 51, 5)

In [14]:
# save the resulting delay array
np.save(os.path.join(mech_dir, 'total_perturbed_mech_delays.npy'), all_delays)

In [10]:
# Also compile the base delays into a giant 1 x (12 * K) array
total_base_delays = np.zeros((N, 5))
table_index = 24
table_dir = os.path.join(mech_dir, f'table_{table_index:04}')
base_delay_file = os.path.join(table_dir, f'base_delays_{table_index:04}.npy')
if not os.path.exists(base_delay_file):
    print(f'Missing base delay file {base_delay_file}')
    raise OSError(f'Missing base delay file {base_delay_file}')

total_base_delays = np.load(base_delay_file)


In [11]:
# save the resulting base delay array
np.save(os.path.join(mech_dir, 'total_base_delays.npy'), total_base_delays)